In [1]:
# Standard library imports
import sys
import argparse
import time

# Third-party imports
import yaml
import awkward as ak
import torch
import torch.nn as nn
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
import wandb
import pennylane as qml

# Local imports
from source.data.opendata import JetNetEvents, TopQuarkEvents
from source.data.datamodule import JetGraphDataModule, JetTorchDataModule
from source.models.qcgnn import QuantumRotQCGNN
from source.models.mpgnn import ClassicalMPGNN
from source.models.pfn import ParticleFlowNetwork
from source.models.part import ParticleTransformer
from source.models.pnet import ParticleNet
from source.training.litmodel import TorchLightningModule, GraphLightningModel
from source.training.loggers import wandb_logger, csv_logger

In [2]:
# Check Python environment and CUDA availability
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is NOT available in this notebook environment!")

Python executable: c:\Users\omviz\.conda\envs\quantum\python.exe
Python version: 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 12:58:53) [MSC v.1929 64 bit (AMD64)]
PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA version: 12.4
GPU device: NVIDIA GeForce RTX 3080


In [3]:
if 'ipykernel' in sys.argv[0]:
    # Jupyter notebook (.ipynb)
    # dataset = 'TopQCD'
    dataset = 'JetNet'
    random_seed = 42
    parser_args = None

else:
    # Python script (.py)
    parser = argparse.ArgumentParser(description='Process some inputs.')
    parser.add_argument('--dataset', type=str, default=None, help='Dataset (default: None)')
    parser.add_argument('--random_seed', type=int, default=42, help='Random seed (default: 42)')
    parser.add_argument('--num_train', type=int, default=None, help='Number of training data (default: None)')
    parser.add_argument('--suffix', type=str, default=None, help='Suffix (default: None)')
    parser_args = parser.parse_args()

    dataset = parser_args.dataset
    random_seed = parser_args.random_seed

with open(f"configs/config.yaml", 'r') as file:
    
    # Configuration of training.
    config = yaml.safe_load(file)
    config['date'] = time.strftime('%Y%m%d_%H%M%S', time.localtime())
    config['dataset'] = dataset

    # Whether change default number of data
    if parser_args is not None:
        if parser_args.num_train is not None:
            config['Data']['num_train'] = parser_args.num_train
            config['Data']['num_val'] = int(0.1 * parser_args.num_train)
            config['Data']['num_test'] = int(0.1 * parser_args.num_train)
        if parser_args.suffix is not None:
            config['Settings']['suffix'] = parser_args.suffix

    # Determine the dimension of the score.
    if config[dataset]['num_classes'] <= 2:
        # Will use `BCELossWithLogits`
        score_dim = 1
    else:
        # Will use `CrossEntropyLoss`
        score_dim = config[dataset]['num_classes']

In [4]:
# Configure GPU and CPU resource limits (80% usage)
import os

# Set GPU memory fraction to 80%
if torch.cuda.is_available():
    gpu_memory_fraction = config.get('Performance', {}).get('gpu_memory_fraction', 0.8)
    
    # Set memory limit for all available GPUs
    for i in range(torch.cuda.device_count()):
        torch.cuda.set_per_process_memory_fraction(gpu_memory_fraction, device=i)
    
    print(f"GPU memory limited to {gpu_memory_fraction*100:.0f}% per device")
    print(f"Available GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total_memory = props.total_memory / (1024**3)  # Convert to GB
        usable_memory = total_memory * gpu_memory_fraction
        print(f"  GPU {i}: {props.name} - Total: {total_memory:.2f} GB, Usable: {usable_memory:.2f} GB")

# Configure CPU worker count based on fraction
cpu_worker_fraction = config.get('Performance', {}).get('cpu_worker_fraction', 0.8)
max_workers = max(1, int((os.cpu_count() or 1) * cpu_worker_fraction))
config['Performance']['num_workers'] = max_workers

print(f"CPU cores: {os.cpu_count()}, Data loader workers: {max_workers} ({cpu_worker_fraction*100:.0f}%)")

GPU memory limited to 80% per device
Available GPUs: 1
  GPU 0: NVIDIA GeForce RTX 3080 - Total: 12.00 GB, Usable: 9.60 GB
CPU cores: 24, Data loader workers: 19 (80%)


In [5]:
def create_data_module(graph: bool, pi_scale: bool = False) -> L.LightningDataModule:
    """Randomly create a data module."""
    
    # Read jet data (reading is not affected by random seed).
    num_train = config['Data']['num_train']
    num_valid = config['Data']['num_valid']
    num_test = config['Data']['num_test']
    num_events = num_train + num_valid + num_test

    # Dataset settings.
    dataset_config = {}
    dataset_config.update(config['Data'])
    dataset_config.update(config[dataset])

    # JetNet dataset (multi-class classification).
    if dataset == 'JetNet':
        channels = ['q', 'g', 't', 'w', 'z']
        events = [JetNetEvents(channel=channel, **dataset_config) for channel in channels]
        events = [_events.generate_events(num_events) for _events in events]

    # Top quark tagging dataset (background light quark QCD).
    elif dataset == 'TopQCD':
        events = []
        for y, channel in enumerate(['Top', 'QCD']):
            events_train = TopQuarkEvents(mode='train', is_signal_new=y, **dataset_config).generate_events(num_train)
            events_valid = TopQuarkEvents(mode='valid', is_signal_new=y, **dataset_config).generate_events(num_valid)
            events_test  = TopQuarkEvents(mode='test',  is_signal_new=y, **dataset_config).generate_events(num_test)
            events.append(ak.concatenate([events_train, events_valid, events_test], axis=0))
    
    # Turn into data module (for lightning training module).
    if graph:
        data_module = JetGraphDataModule(events, pi_scale=pi_scale, **dataset_config)
    else:
        data_module = JetTorchDataModule(events, **dataset_config)

    return data_module


def create_training_info(model: nn.Module, model_description: str, model_hparams: dict, lr: float) -> dict:
    """Create a training information dictionary that will be recorded."""

    # Data information.
    data_description = f"{dataset}_P{config['Data']['max_num_ptcs']}_N{config['Data']['num_train']}"

    # The description used for grouping different result of random seeds.
    group_rnd = '-'.join([model.__class__.__name__, model_description, data_description])
    
    # Name for this particular training (including random seed).
    name = group_rnd
    if config['Settings']['suffix'] != '':
        name += '-' + config['Settings']['suffix']
    name += '-' + str(random_seed)

    # Hyperparameters and configurations that will be recorded.
    training_info = config.copy()
    training_info.update(model_hparams)
    training_info.update({
        'lr': lr,
        'name': name,
        'date': config['date'],
        'model': model.__class__.__name__,
        'group_rnd': group_rnd,
        'random_seed': random_seed,
        'data_description': data_description,
        'model_description': model_description,
    })

    return training_info

    
def create_lightning_model(model: nn.Module, graph: bool, lr: float) -> L.LightningModule:
    """Create a lightning model for trainer."""

    # Optimizer.
    optimizer = torch.optim.RAdam(model.parameters(), lr=lr)

    # Create lightning model depends on graph or not.
    print_log = config['Settings']['print_log']

    # Graph is for PFN.
    if graph:
        return GraphLightningModel(model, optimizer=optimizer, score_dim=score_dim, print_log=print_log)
    else:
        return TorchLightningModule(model, optimizer=optimizer, score_dim=score_dim, print_log=print_log)


def create_trainer(model: nn.Module, training_info: dict, accelerator: str) -> L.Trainer:
    """Create lightning trainer for training with optimized GPU/CPU usage."""

    # Determine the number of devices to use based on accelerator type
    if accelerator == 'gpu':
        # Use all available GPUs
        devices = torch.cuda.device_count() if torch.cuda.is_available() else 1
        # Enable optimizations for GPU
        precision = '16-mixed' if torch.cuda.is_available() else '32'
    else:
        # Use all available CPU cores
        import os
        devices = 1  # CPU doesn't benefit from multiple devices in Lightning
        precision = '32'
    
    print(f"Training on {accelerator} with {devices} device(s), precision: {precision}")

    # Create logger for monitoring the training.
    if config['Settings']['use_wandb']:
        import os
        # Prefer using an existing WANDB_API_KEY if available. Otherwise try anonymous login
        # (supported in newer wandb versions), and finally fall back to offline/csv logging.
        api_key = os.environ.get('WANDB_API_KEY', '')
        try:
            if api_key:
                # normal login with API key
                wandb.login()
            else:
                # try anonymous login first; if the installed wandb doesn't accept `anonymous`,
                # an exception will be raised and we'll fallback to offline mode.
                try:
                    wandb.login(anonymous='allow')
                except TypeError:
                    # older wandb version without anonymous support
                    os.environ['WANDB_MODE'] = 'offline'
                    wandb.login()

            logger = wandb_logger(training_info)
            # It's safe to call watch only when logger exists and wandb didn't fail.
            try:
                logger.watch(model)
            except Exception:
                # some logger wrappers don't implement watch; ignore silently
                pass
            loggers = [logger, csv_logger(training_info)]

        except Exception as e:
            # If wandb login fails for any reason, continue with a local CSV logger and
            # set WANDB_MODE=offline to prevent further network prompts.
            print(f"wandb login failed ({e}); falling back to local csv logger and offline mode.")
            os.environ['WANDB_MODE'] = 'offline'
            loggers = csv_logger(training_info)

    else:
        loggers = csv_logger(training_info)
    
    # Return the lightning trainer with performance optimizations.
    return L.Trainer(
        logger=loggers,
        accelerator=accelerator,
        devices=devices,
        precision=precision,
        max_epochs=config['Train']['max_epochs'],
        log_every_n_steps=config['Train']['log_every_n_steps'],
        num_sanity_val_steps=config['Train']['num_sanity_val_steps'],
        # Performance optimizations
        benchmark=True,  # cudnn.benchmark for faster training
        deterministic=False,  # Allow non-deterministic operations for speed
        # Enable gradient accumulation if memory is limited
        # accumulate_grad_batches=1,
        callbacks=[ModelCheckpoint(
            monitor=config['Train']['ckpt_monitor'],
            mode=config['Train']['ckpt_mode'],
            save_top_k=config['Train']['ckpt_top_k'],
            save_last=True,
            filename='{epoch}-{valid_auc:.3f}-{valid_accuracy:.3f}',
        )],
    )

In [6]:
def train(
        model: nn.Module, model_description: str, model_hparams: dict,
        accelerator: str, lr: float, graph: bool, pi_scale: bool = False
    ):
    
    # Fix all random stuff.
    L.seed_everything(random_seed)

    # Traditional training procedure.
    data_module = create_data_module(graph=graph, pi_scale=pi_scale)
    lightning_model = create_lightning_model(model=model, graph=graph, lr=lr)
    training_info = create_training_info(model, model_description, model_hparams, lr=lr)
    trainer = create_trainer(model, training_info, accelerator)
    
    # Training and validation.
    if eval(config['Settings']['mode'][0]):
        if eval(config['Settings']['mode'][1]):
            trainer.fit(lightning_model, datamodule=data_module)
        else:
            trainer.fit(lightning_model, train_dataloaders=data_module.train_dataloader())

    # Testing.
    if eval(config['Settings']['mode'][2]):
        trainer.test(lightning_model, datamodule=data_module, ckpt_path='best')

    # Finish wandb if used.
    if config['Settings']['use_wandb']:
        wandb.finish()

    return training_info['name']

def train_quantum(model_class: nn.Module, model_hparams: dict, pi_scale: bool, lr: float):

    model = model_class(score_dim=score_dim, **model_hparams)

    num_ir_qubits = model_hparams['num_ir_qubits']
    num_nr_qubits = model_hparams['num_nr_qubits']
    num_layers = model_hparams['num_layers']
    num_reupload = model_hparams['num_reupload']
    dropout = model_hparams['dropout']
    model_description = f"nI{num_ir_qubits}_nQ{num_nr_qubits}_L{num_layers}_R{num_reupload}_D{dropout:.2f}"

    accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'
    name = train(model, model_description, model_hparams, accelerator, lr=lr, graph=False, pi_scale=pi_scale)

    return name

def train_mpgnn(model_hparams: dict, lr: float):
    
    model = ClassicalMPGNN(score_dim=score_dim, **model_hparams)

    phi_out = model_hparams['phi_out']
    phi_hidden = model_hparams['phi_hidden']
    phi_layers = model_hparams['phi_layers']
    dropout = model_hparams['dropout']
    model_description = f"O{phi_out}_H{phi_hidden}_L{phi_layers}_D{dropout:.2f}"

    accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'
    name = train(model, model_description, model_hparams, accelerator, lr=lr, graph=True)
    
    return name

def train_benchmark(model_class: nn.Module, lr: float):
    with open('configs/benchmark.yaml', 'r') as file:
        hparams = yaml.safe_load(file)[model_class.__name__]
        model_description = ''

    model = model_class(score_dim=score_dim, parameters=hparams)
    
    accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'

    if model_class == ParticleFlowNetwork:
        name = train(model, model_description, hparams, accelerator, lr=lr, graph=True)
    else:
        name = train(model, model_description, hparams, accelerator, lr=lr, graph=False)
    
    return name

In [ ]:
# QCGNN
for Q in [3, 6]:
    print(f"\n* Train QCGNN n_Q = {Q}.\n")
    qcgnn_hparams = {'num_ir_qubits': 4, 'num_nr_qubits': Q, 'num_layers': Q // 3, 'num_reupload': 2, 'dropout': 0.0, 'vqc_ansatz': qml.StronglyEntanglingLayers}
    name = train_quantum(model_class=QuantumRotQCGNN, model_hparams=qcgnn_hparams, pi_scale=True, lr=1e-3)


* Train QCGNN n_Q = 3.



Seed set to 42


Loading JetNet dataset from channel: q
Loading JetNet dataset from channel: g
Loading JetNet dataset from channel: t
Loading JetNet dataset from channel: w
Loading JetNet dataset from channel: z


c:\Users\omviz\.conda\envs\quantum\Lib\site-packages\lightning\fabric\loggers\csv_logs.py:268: Experiment logs directory d:\BACKUP\code\Quantum\QuantumGNN-\Jet\QCGNN\training_logs\CSVLogger\QuantumRotQCGNN-nI4_nQ3_L1_R2_D0.00-JetNet_P16_N2500-42\lastest_run exists and is not empty. Previous log files in this directory will be deleted when the new ones are saved!
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3080') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\omviz\.conda\envs\quantum\Lib\site-packages\lightning\pytorc

Training on gpu with 1 device(s), precision: 16-mixed


In [ ]:
# MPGNN (3, 6)
for phi_dim in [3, 6]:
    pfn_hparams = {'phi_in': 3 + 3, 'phi_out': phi_dim, 'phi_layers': 2, 'phi_hidden': phi_dim, 'mlp_hidden': 16, 'dropout': 0.0}
    name = train_mpgnn(model_hparams=pfn_hparams, lr=1e-3)

# MPGNN (64)
pfn_hparams = {'phi_in': 3 + 3, 'phi_out': 64, 'phi_layers': 2, 'phi_hidden': 64, 'mlp_hidden': 64, 'dropout': 0.0}
name = train_mpgnn(model_hparams=pfn_hparams, lr=1e-3)

# Particle Flow Network.
print(f"\n* Train Particle Flow Network.\n")
name = train_benchmark(model_class=ParticleFlowNetwork, lr=1e-3)

# Particle Transformer.
print(f"\n* Train Particle Transformer.\n")
name = train_benchmark(model_class=ParticleTransformer, lr=1e-3)

# Particle Net.
print(f"\n* Train Particle Net.\n")
name = train_benchmark(model_class=ParticleNet, lr=1e-3)